# recipe-dataclass — worked example 3: Recipe for sum_forward with a dim kwarg

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `recipe-dataclass`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When a forward op takes a non-Tensor configuration argument (like `dim` for a reduction), that value belongs in `kwargs`, NOT in `parents`. Only Tensor inputs that need gradients go into `parents`. The kwargs are replayed verbatim into the backward call, so the keys must match the backward function's signature.

## Worked solution

`sum_forward(x, *, dim)` reduces `x.array` along `dim` with `keepdim=True` so the shape is recoverable in backward. We build the `Recipe` with `func = t.sum`, `args = (x.array,)` (just the one unboxed Tensor), and `kwargs = {'dim': dim, 'keepdim': True}` — these are the exact keys `torch.sum` and its backward rule expect. The `dim` integer is configuration, not a differentiable input, so it does NOT go into `parents`. `parents` is `{0: x}`: only the Tensor at argnum 0. This separation is the whole point — `parents` drives graph traversal, `kwargs` replays the call.

In [ ]:
from dataclasses import dataclass
from typing import Callable


class MiniTensor:
    def __init__(self, array, recipe=None):
        self.array = array
        self.recipe = recipe
        self.grad = None


@dataclass
class Recipe:
    func: Callable
    args: tuple
    kwargs: dict
    parents: dict


def sum_forward(x: MiniTensor, *, dim: int) -> MiniTensor:
    out_arr = x.array.sum(dim=dim, keepdim=True)
    recipe = Recipe(
        func=t.sum,
        args=(x.array,),
        kwargs={'dim': dim, 'keepdim': True},
        parents={0: x},
    )
    return MiniTensor(out_arr, recipe=recipe)


x = MiniTensor(t.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]))
out = sum_forward(x, dim=1)
print('out shape:', tuple(out.array.shape))
print('kwargs:', out.recipe.kwargs)
print('parents keys:', list(out.recipe.parents.keys()))